<h1 style="text-align: center; margin-bottom: 0;">Deep Learning based Recommendation System</h1>
<h2 style="text-align: center; margin-top: 35px;">Sravya Kodati, Rutvik Dhopate, Vivek Rachakonda</h2>

<center  style="font-size:22px; margin-top: 50px;">NRCMA - Neural Recommendation with Cross-Modality Mutual Attention (Luo, et al.)</center>


<ul style="font-size:20px;">
    <li>Recap of problem statement</li>
    <li>Architecture</li>
    <li>Libraries</li>
    <li>Preprocessing</li>
    <li>PyTorch Model Class</li>
    <li>Training Loop</li>
    <li>Next Steps</li>
</ul>

## Architecture

<details>
    <div style="text-align: center;">
    <img src="../assets/images/nrcma_architecture.png"></img>   
    </div>
</details>

## Libraries

In [201]:
import math
import yaml
import torch
import random

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import numpy as np
import pandas as pd
from torchinfo import summary
import matplotlib.pyplot as plt
from dataclasses import dataclass
from tqdm import tqdm

import gensim.downloader as api
from sklearn.model_selection import train_test_split

In [156]:
df = pd.read_csv("../data/books_mini_sample.csv")

In [158]:
filtered_df = df[['rating', 'text', 'user_id', 'asin']].copy()
filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,AE224GVO7OHTYF26U6ER6BEVIUAQ,0486241386
1,5.0,Ladies 'n Gentlemen...we have another hit from...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000MGATTE
2,5.0,I ordered this book for my Kindle. Koontz and ...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UDNBRQ
3,5.0,Absolutely loved this book! Stephen King just...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B000UZJREU
4,5.0,Another King satisfied reader~! what can I sa...,AE225Z2VRWT6GPTOMA4H4O3H2KVQ,B001RF3U9K


In [159]:
filtered_df['user_id'] = filtered_df['user_id'].astype('category').cat.codes
filtered_df['asin'] = filtered_df['asin'].astype('category').cat.codes

filtered_df.head()

,rating,text,user_id,asin
0,5.0,This is a great book with a lot of detail. ON...,0,1655
1,5.0,Ladies 'n Gentlemen...we have another hit from...,1,5766
2,5.0,I ordered this book for my Kindle. Koontz and ...,1,5817
3,5.0,Absolutely loved this book! Stephen King just...,1,5819
4,5.0,Another King satisfied reader~! what can I sa...,1,5877


### Load Glove vectors

In [161]:
word2vec_model = api.load('word2vec-google-news-300')

# Access word embeddings from the pre-trained model
vector = word2vec_model['word']

In [162]:
# adding a padding token
word2vec_model['<pad>'] = np.zeros(300)

## Preprocessing

In [163]:
%%time
max_length = 100
filtered_df['text'] = filtered_df['text'].astype('string')\
                                        .fillna('')\
                                        .str.split(' ')\
                                        .apply(lambda words: [word for word in words if word in word2vec_model.key_to_index])

filtered_df['text'] = filtered_df['text'].apply(
    lambda x: x[:max_length] + ['<pad>'] * (max_length - len(x)) if len(x) < max_length else x[:max_length]
)

CPU times: user 1.33 s, sys: 489 ms, total: 1.82 s
Wall time: 3.67 s


In [164]:
filtered_df['rating'] = (filtered_df['rating'] - filtered_df['rating'].min()) / (filtered_df['rating'].max() - filtered_df['rating'].min())

In [165]:
filtered_df[['text', 'rating']].head()

,text,rating
0,"[This, is, great, book, with, lot, ONE, CAUTIO...",1.0
1,"[Ladies, have, another, hit, from, Stephen, th...",1.0
2,"[I, ordered, this, book, for, my, Koontz, two,...",1.0
3,"[Absolutely, loved, this, Stephen, King, just,...",1.0
4,"[Another, King, satisfied, what, can, I, The, ...",1.0


## Loading config file

In [166]:
# Load Configuration

with open('../config/nrcma.yaml') as f:
    config = yaml.safe_load(f)

### Fix all the seeds for reproducibility

In [167]:
def seed_everything(seed=42):
    
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    # this may hurt the performance
    torch.backends.cudnn.benchmark = False


# Fix all the seeds for reproducibility
seed_everything(seed=config['training']['random_seed'])

### WandB for logging and checkpointing

In [187]:
import wandb

wandb_project = 'deep_rec_sys_user_reviews'

In [207]:
run = wandb.init(project=wandb_project, 
                 config=config, 
                 tags=["run_with_batches", "testing_multiple_batches"],
                 name="nrcma_run_4",
                 notes="This is the fourth run of NRCMA! Trying to see if we can fit and learn 100 batches!!")

## PyTorch Dataset Class

In [174]:
%%time
from torch.utils.data import Dataset, DataLoader


class RecSysDataset(Dataset):
    def __init__(self, df, word2vec_model, max_reviews):
        self.df = df
        self.word2vec_model = word2vec_model
        self.max_reviews = max_reviews

    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        user_reviews = self.df[self.df['user_id'] == row.user_id]['text']
        w2v_user_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in user_reviews if len(i)!=0]
        user_tower_input = torch.stack(w2v_user_tensor, dim=0)

        item_reviews = self.df[self.df['asin'] == row.asin]['text']
        w2v_item_tensor = [torch.from_numpy(self.word2vec_model[i]) for i in item_reviews if len(i)!=0]
        item_tower_input = torch.stack(w2v_item_tensor, dim=0)

        target_size = (self.max_reviews, max_length, 300)
        user_tower_input = self._resize_tensors(user_tower_input, target_size)
        item_tower_input = self._resize_tensors(item_tower_input, target_size)
        
        user_id = torch.tensor([row['user_id']], dtype=torch.int).squeeze(-1)
        asin = torch.tensor([row['asin']], dtype=torch.int).squeeze(-1)
        rating = torch.tensor([row['rating']], dtype=torch.float).squeeze(-1)

        return user_tower_input, item_tower_input, rating, user_id, asin

    def _resize_tensors(self, input_tensor, target_size):

        # Resize the num_reviews if needed; TODO - change to random sampling? picking only first `max_reviews` number of reviews of the user or item
        tensor_size = input_tensor.shape
        if tensor_size[0] != target_size[0]:
            if tensor_size[0] < target_size[0]:
                diff = target_size[0] - tensor_size[0]
                padding = torch.zeros((diff, tensor_size[1], tensor_size[2]))
                output_tensor = torch.cat((input_tensor, padding), dim=0)
            else:
                output_tensor = input_tensor[:target_size[0]]
        else:
            output_tensor = input_tensor
        return output_tensor



CPU times: user 81 μs, sys: 113 μs, total: 194 μs
Wall time: 208 μs


In [175]:
filtered_df.columns

Index(['rating', 'text', 'user_id', 'asin'], dtype='object')

In [202]:
%%time

# lets split the data into train, test, val - its 2025 and sklearn still doesn't support val split in its function!! :(
X_train, X_test = train_test_split(filtered_df, test_size=0.05, random_state=42)
X_train, X_val = train_test_split(X_train, test_size=(0.05/0.95), random_state=42)

train_recsys_dataset = RecSysDataset(X_train, word2vec_model, 10)
val_recsys_dataset = RecSysDataset(X_val, word2vec_model, 10)
test_recsys_dataset = RecSysDataset(X_test, word2vec_model, 10)

train_dataloader = DataLoader(train_recsys_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_recsys_dataset, batch_size=32, shuffle=True)

for i, sample in enumerate(train_dataloader):
    if i > 2:
        break
    user_tower_input, item_tower_input, label, user_id, asin = sample
    print(user_tower_input.shape, item_tower_input.shape, label.shape, user_id.shape, asin.shape)
    

torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300]) torch.Size([32]) torch.Size([32]) torch.Size([32])
torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300]) torch.Size([32]) torch.Size([32]) torch.Size([32])
torch.Size([32, 10, 100, 300]) torch.Size([32, 10, 100, 300]) torch.Size([32]) torch.Size([32]) torch.Size([32])
CPU times: user 3.75 s, sys: 829 ms, total: 4.58 s
Wall time: 1.29 s


## NRCMA Module

In [203]:
# creating a dataclass for the model

@dataclass
class NRCMAConfig:
    num_users: int
    num_products: int
    num_conv_filters: int
    kernel_size: int
    word_embed_dim: int
    id_embed_dim: int
    attention_vector_dim: int
    feature_vector_dim: int

    @staticmethod
    def from_config(config):
        return NRCMAConfig(**config)


In [204]:
nrcma_config = NRCMAConfig.from_config(config['model'])

In [181]:
class NRCMA(nn.Module):
    def __init__(self, config: NRCMAConfig):
        super(NRCMA, self).__init__()

        # Extract all config parameters into local variables
        self.num_users = config.num_users
        self.num_products = config.num_products
        self.id_embed_dim = config.id_embed_dim
        self.num_conv_filters = config.num_conv_filters
        self.kernel_size = config.kernel_size
        self.word_embed_dim = config.word_embed_dim
        self.attention_vector_dim = config.attention_vector_dim
        self.feature_vector_dim = config.feature_vector_dim
        
        self.user_embedding = nn.Embedding(self.num_users, self.id_embed_dim)
        self.item_embedding = nn.Embedding(self.num_products, self.id_embed_dim)

        # user tower
        self.user_cnn = nn.Conv1d(in_channels=self.word_embed_dim, 
                                  out_channels=self.num_conv_filters, 
                                  kernel_size=self.kernel_size, 
                                  padding='same', 
                                  bias=True)
        self.word_level_matrix = nn.Linear(self.id_embed_dim, self.attention_vector_dim)
        self.word_harmony_matrix = nn.Linear(self.num_conv_filters, self.attention_vector_dim)
        self.review_level_matrix = nn.Linear(self.id_embed_dim, self.attention_vector_dim)
        self.review_harmony_matrix = nn.Linear(self.num_conv_filters, self.attention_vector_dim)

        # item tower
        self.item_cnn = nn.Conv1d(in_channels=self.word_embed_dim, 
                                  out_channels=self.num_conv_filters, 
                                  kernel_size=self.kernel_size, 
                                  padding='same', 
                                  bias=True)
        self.item_word_level_matrix = nn.Linear(self.id_embed_dim, self.attention_vector_dim)
        self.item_word_harmony_matrix = nn.Linear(self.num_conv_filters, self.attention_vector_dim)
        self.item_review_level_matrix = nn.Linear(self.id_embed_dim, self.attention_vector_dim)
        self.item_review_harmony_matrix = nn.Linear(self.num_conv_filters, self.attention_vector_dim)

        # factorization machine, 2 * num_conv_filters as we get concat features from user, item
        self.fm_linear = nn.Linear(2*self.num_conv_filters, 1)
        self.v = nn.Parameter(torch.empty(2*self.num_conv_filters, self.feature_vector_dim))
        
        # Initialize self.v similar to nn.Linear weights
        nn.init.kaiming_uniform_(self.v, a=math.sqrt(5))


    def forward(self, user_input, item_input, user_id, item_id):
        
        user_embedding = self.user_embedding(user_id)
        item_embedding = self.item_embedding(item_id)

        d_u = self.process_single_tower(user_input, item_embedding, tower='user')
        d_i = self.process_single_tower(item_input, user_embedding, tower='item')
        o = torch.cat((d_u, d_i), dim=1)

        # factorization machine
        linear_part = self.fm_linear(o).squeeze(1) 
        interaction = o.unsqueeze(2) * self.v
        square_of_sum = (interaction.sum(dim=1) ** 2)  
        sum_of_square = (interaction ** 2).sum(dim=1)

        interaction_part = 0.5 * (square_of_sum - sum_of_square).sum(dim=1)
        prediction = linear_part + interaction_part
        
        return prediction

    def process_single_tower(self, input_matrix, embedding, tower):

        if tower == 'user':
            cnn = self.user_cnn
            word_level_matrix = self.word_level_matrix
            word_harmony_matrix = self.word_harmony_matrix
            review_level_matrix = self.review_level_matrix
            review_harmony_matrix = self.review_harmony_matrix
        else:
            cnn = self.item_cnn
            word_level_matrix = self.item_word_level_matrix
            word_harmony_matrix = self.item_word_harmony_matrix
            review_level_matrix = self.item_review_level_matrix
            review_harmony_matrix = self.item_review_harmony_matrix

        # process information through single tower - employing cross attention
        batch_size = input_matrix.shape[0]
        input_matrix = input_matrix.flatten(start_dim=0, end_dim=1).transpose(1, 2)
        review_features = F.relu(cnn(input_matrix))
        beta_k = F.relu(word_level_matrix(embedding))
        
        review_features = review_features.view(batch_size, -1, *review_features.shape[1:])
        
        intermediate_output_1 = word_harmony_matrix(review_features.transpose(-1, -2)).transpose(-1, -2) # not sure if this step is correct; need to check this
        beta_k = beta_k.unsqueeze(1)
        beta_k = beta_k.unsqueeze(2)
        
        b_c = beta_k @ intermediate_output_1
        b_c = b_c.squeeze(2)
        alpha_c = nn.Softmax(dim=-1)(b_c)
        
        d_uk = alpha_c.unsqueeze(2) @ review_features.transpose(-1, -2)
        d_uk = d_uk.squeeze(2)

        beta_u = F.relu(review_level_matrix(embedding))
        intermediate_output_2 = review_harmony_matrix(d_uk)
        b_k = beta_u.unsqueeze(1) @ intermediate_output_2.transpose(1, 2)

        alpha_k = nn.Softmax(dim=-1)(b_k)
        d_u = alpha_k @ d_uk
        d_u = d_u.squeeze(1)

        return d_u

In [182]:
model = NRCMA(nrcma_config)

## Model Summary

<span style="font-size:20px;">92.5% of parameters are present in the embedding layers</span>

In [183]:
summary(model)

Layer (type:depth-idx)                   Param #
NRCMA                                    1,600
├─Embedding: 1-1                         491,940
├─Embedding: 1-2                         236,670
├─Conv1d: 1-3                            24,080
├─Linear: 1-4                            620
├─Linear: 1-5                            1,620
├─Linear: 1-6                            620
├─Linear: 1-7                            1,620
├─Conv1d: 1-8                            24,080
├─Linear: 1-9                            620
├─Linear: 1-10                           1,620
├─Linear: 1-11                           620
├─Linear: 1-12                           1,620
├─Linear: 1-13                           161
Total params: 787,491
Trainable params: 787,491
Non-trainable params: 0

In [184]:
config['training']

{'batch_size': 64,
 'learning_rate': 0.001,
 'num_epochs': 10,
 'random_seed': 42,
 'log_interval': 10,
 'checkpoint_dir': 'checkpoints/'}

In [208]:
args = config['training']

device = torch.device("mps")

model = NRCMA(nrcma_config)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=args['learning_rate'])

loss_fn = nn.MSELoss()

## PyTorch Training Loop

In [209]:
%%time
total_loss = []
max_batches = 200
# args['num_epochs'] = 10 # TODO - remove this line; only for testing

for epoch in tqdm(range(args['num_epochs'])):

    model.train()
    for i, sample in enumerate(train_dataloader):
        if i > max_batches:
            break
        optimizer.zero_grad(set_to_none=True)
        user_tower_input, item_tower_input, true_rating, user_id, asin = sample
        user_tower_input, item_tower_input, true_rating, user_id, asin = user_tower_input.to(device), item_tower_input.to(device), true_rating.to(device), user_id.to(device), asin.to(device)
        predicted_rating = model(user_tower_input, item_tower_input, user_id, asin)
        train_loss = loss_fn(predicted_rating, true_rating)

        train_loss.backward()
        optimizer.step()
        total_loss.append(train_loss.item())

        if i < max_batches:
            run.log({"train/train_loss":train_loss.item(), "train/epoch": epoch})

    model.eval()
    val_loss = 0
    for j, sample in enumerate(val_dataloader):
        user_tower_input, item_tower_input, true_rating, user_id, asin = sample
        user_tower_input, item_tower_input, true_rating, user_id, asin = user_tower_input.to(device), item_tower_input.to(device), true_rating.to(device), user_id.to(device), asin.to(device)
        prediction = model(user_tower_input, item_tower_input, user_id, asin)
        loss = loss_fn(prediction, true_rating)
        val_loss += loss.item()

    run.log({"train/train_loss":train_loss.item(), "train/epoch":epoch, "val/val_loss":val_loss/len(val_dataloader)})

    print(f'avg val loss - {val_loss/len(val_recsys_dataset)}')


  3%|███▍                                                                                                  | 1/30 [01:08<33:06, 68.50s/it]

avg val loss - 0.0017237585624263385


  7%|██████▊                                                                                               | 2/30 [02:18<32:15, 69.12s/it]

avg val loss - 0.001500192824009455


 10%|██████████▏                                                                                           | 3/30 [03:29<31:35, 70.20s/it]

...

avg val loss - 0.0014597396159786776


 17%|█████████████████                                                                                     | 5/30 [06:16<33:04, 79.38s/it]

avg val loss - 0.0014363196076500819


 17%|█████████████████                                                                                     | 5/30 [06:20<31:42, 76.12s/it]


KeyboardInterrupt: 

In [194]:
run.finish()

train/epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/train_loss,█▇▇█▆▅▅▃▃▃▂▂▃▄▄▄▃▂▂▃▃▃▃▂▂▂▂▂▂▁▂▂▁▁▁▂▁▁▁▁
train/epoch,29
train/train_loss,0.05127


In [199]:
len(val_dataloader)

166

In [210]:
torch.save({
        # 'epoch': current_epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, 'nrcma_fourth_run.pt')
run.log_model("./nrcma_fourth_run.pt", "nrcma_fourth_run")
run.finish()

train/epoch,▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▇▇▇▇▇▇▇▇█
train/train_loss,█▃▂▂▂▂▂▂▁▂▂▁▁▁▂▁▁▁▂▁▁▁▁▁▁▂▂▁▁▁▂▁▁▂▂▂▂▁▂▁
val/val_loss,█▃▂▂▁
train/epoch,5
train/train_loss,0.03188
val/val_loss,0.04585


## Next Steps

<ul style="font-size:18px;">
    <li>Split this notebook into 3 scripts - utils (data loading), model class, training</li>
    <li>Retraining - starting from a intermediate checkpoint</li>
    <li>Perform Hyperparameter Tuning </br>
        - conv filters </br>
        - attention vector dimension
    </li>
    <li>Train the model on GPU</li>
    <li>Scaling the training to multiple GPUs</li>
</ul>

## Adding batch dimension!!

In [119]:

# num_users, reviews, words, word_embedding
test_input = torch.randn(6, 8, 5, 3)
test_input = test_input.flatten(start_dim=0, end_dim=1) # num_users * reviews, words, word_embedding
test_input = test_input.transpose(1, 2) # num_users * reviews, word_embedding, words

user_cnn = nn.Conv1d(in_channels=3, out_channels=10, kernel_size=2, padding='same') # out_channels, in_channels, kernel_size

cnn_output = user_cnn(test_input)
print(f'cnn_output shape - ', cnn_output.shape) # num_users * reviews, out_channels, words

user_embeddings = nn.Embedding(100, 7)
user_ids = torch.randint(0, 6, (6,)) # num_users, 

q = user_embeddings(user_ids).T # id_embed_dim, num_users
print('user_embeddings(q) shape - ', q.shape)

word_level = nn.Linear(in_features=7, out_features=9) # attention_vector_dim = 9, id_embed_dim = 7
beta_k = word_level(q.T)
print(f'beta_k shape - ', beta_k.shape) # num_users, attention_vector_dim

# time to split users
cnn_output = cnn_output.view(6, 8, 10, 5) # num_users, reviews, out_channels, words
print(f'updated cnn_output shape - ', cnn_output.shape)

word_harmony = nn.Linear(in_features=10, out_features=9) # attention_vector_dim, out_channels
intermediate_output = word_harmony(cnn_output.transpose(2, 3)).transpose(2, 3)
print(f'intermediate_output shape - ', intermediate_output.shape) # num_users, reviews, attention_vector_dim, words

beta_k = beta_k.unsqueeze(1)
beta_k = beta_k.unsqueeze(2)
print(f'modified beta_k shape - ', beta_k.shape)

b_c = beta_k @ intermediate_output # num_users, reviews, 1, words
b_c = b_c.squeeze(2) # num_users, reviews, words
print(f'b_c shape - ', b_c.shape)

alpha_c = nn.Softmax(dim=-1)(b_c)
print(f'alpha_c shape - ', alpha_c.shape) # num_users, reviews, words

d_uk = alpha_c.unsqueeze(2) @ cnn_output.transpose(2, 3)
print(f'd_uk shape - ', d_uk.shape) # num_users, reviews, 1, out_channels

d_uk = d_uk.squeeze(2)
print(f'modified d_uk shape - ', d_uk.shape) # num_users, reviews, out_channels

review_level = nn.Linear(in_features=7, out_features=9) # attention_vector_dim=9, id_embed_dim=7
beta_u = review_level(q.T)
print(f'beta_u shape - ', beta_u.shape) # num_user, attention_vector_dim

review_harmony = nn.Linear(in_features=10, out_features=9) # attention_vector_dim, out_channels
intermediate_output_2 = review_harmony(d_uk)
print('intermediate_output_2 shape', intermediate_output_2.shape) # num_users, reviews, attention_vector_dim

b_k = beta_u.unsqueeze(1) @ intermediate_output_2.transpose(1, 2)
print(f'b_k shape - {b_k.shape}') # num_users, 1, reviews

alpha_k = nn.Softmax(dim=-1)(b_k)
print(f'alpha_k shape - {alpha_k.shape}')

d_u = alpha_k @ d_uk # num_users, 1, out_channels
d_u = d_u.squeeze(1)
print(f'd_u shape - {d_u.shape}')

cnn_output shape -  torch.Size([48, 10, 5])
user_embeddings(q) shape -  torch.Size([7, 6])
beta_k shape -  torch.Size([6, 9])
updated cnn_output shape -  torch.Size([6, 8, 10, 5])
intermediate_output shape -  torch.Size([6, 8, 9, 5])
modified beta_k shape -  torch.Size([6, 1, 1, 9])
b_c shape -  torch.Size([6, 8, 5])
alpha_c shape -  torch.Size([6, 8, 5])
d_uk shape -  torch.Size([6, 8, 1, 10])
modified d_uk shape -  torch.Size([6, 8, 10])
beta_u shape -  torch.Size([6, 9])
intermediate_output_2 shape torch.Size([6, 8, 9])
b_k shape - torch.Size([6, 1, 8])
alpha_k shape - torch.Size([6, 1, 8])
d_u shape - torch.Size([6, 10])


In [113]:
a = nn.Linear(5, 3)
b = torch.randn(10, 5)
print(a._parameters['weight'].shape)
print(a(b).shape)

torch.Size([3, 5])
torch.Size([10, 3])


### Factorization Machine

In [140]:
d_u = torch.randn(1, 10)
d_i = torch.randn(1, 10)

o = torch.cat((d_u, d_i), dim=1)
print(o.T.shape)
v = torch.randn(20, 3) # num_features, k (feature_vector_dim)

(((v * o.T)**2).sum(dim=0) - (v**2 * o.T**2).sum(dim=0)).sum()

torch.Size([20, 1])


tensor(7.6294e-06)

In [152]:
import torch

batch_size = 4  # Number of users in the batch
d_u = torch.randn(batch_size, 10)  # users * user_feature
d_i = torch.randn(batch_size, 10)  # items * item_feature

o = torch.cat((d_u, d_i), dim=1)  # Shape: (batch_size, 20)
v = torch.randn(20, 3)  # num_features, k (feature_vector_dim)

# Linear part
w = nn.Linear(20, 1)
linear_part = w(o).squeeze(1)  # (batch_size,)
print(linear_part.shape)

# Interaction part
interaction = o.unsqueeze(2) * v  # (batch_size, 20, 3)
square_of_sum = (interaction.sum(dim=1) ** 2)  # (batch_size, 3)
sum_of_square = (interaction ** 2).sum(dim=1)  # (batch_size, 3)

interaction_part = 0.5 * (square_of_sum - sum_of_square).sum(dim=1)  # (batch_size,)

print(interaction_part.shape)

# FM Output: Linear + Interaction
output = linear_part + interaction_part
print(output.shape)

torch.Size([4])
torch.Size([4])
torch.Size([4])


In [153]:
import torch

x = torch.randn(48, 10, 5)
new_x = x.view(6, -1, *x.shape[1:])  # new shape is (6, 8, 10, 5)
print(new_x.shape)

torch.Size([6, 8, 10, 5])
